# 04. NV4: Đánh giá hiệu suất và Xếp hạng trạm sạc (Station Benchmarking)

## Mục tiêu nghiên cứu (NV4)
1. Đo lường các chỉ số hiệu suất trọng yếu (KPIs) cho từng trạm và cụm trạm sạc:
   - Tần suất sử dụng trạm (Session throughput)
   - Tỷ lệ thời gian sạc hữu ích (Utilization Efficiency = $T_{charging} / T_{total}$)
   - Mức độ tắc nghẽn và chiếm giữ nhàn rỗi trung bình theo trạm
   - Tổng sản lượng điện năng cung cấp ($E$) và doanh thu thu được.
2. Phân tích ma trận 4 góc phần tư (Quadrant Analysis): Nhu cầu cao vs Tỷ lệ tắc nghẽn cao.
3. Đề xuất danh sách trạm cần ưu tiên mở rộng hạ tầng hoặc áp dụng điều tiết phí.


In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.append(str(Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()))
from src.config import SESSIONS_FEATURES_PARQUET
from src.evaluation import compute_station_benchmarks
from src.visualization import set_plotting_style

set_plotting_style()
df = pd.read_parquet(SESSIONS_FEATURES_PARQUET)
station_benchmarks = compute_station_benchmarks(df)
print(f"Tổng số trạm sạc được đánh giá: {len(station_benchmarks):,} trạm")


### 1. Bảng xếp hạng Top 10 trạm sạc bận rộn nhất


In [ ]:
display(station_benchmarks.head(10))


### 2. Phân tích ma trận Quadrant: Tần suất phiên sạc vs Tỷ lệ chiếm dụng nhàn rỗi


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
sns.scatterplot(
    data=station_benchmarks, 
    x='total_sessions', 
    y='avg_idle_ratio', 
    size='total_energy_kwh', 
    sizes=(40, 400), 
    alpha=0.7, 
    color='#1f77b4',
    ax=ax
)
ax.axhline(station_benchmarks['avg_idle_ratio'].median(), color='red', linestyle='--', label='Median Idle Ratio')
ax.axvline(station_benchmarks['total_sessions'].median(), color='green', linestyle='--', label='Median Sessions')
ax.set_title('Ma trận Đánh giá Trạm sạc (Sessions vs Avg Idle Ratio)')
ax.set_xlabel('Tổng số phiên sạc')
ax.set_ylabel('Tỷ lệ nhàn rỗi trung bình (Idle Ratio)')
ax.legend()
plt.show()
